# Notebook 02 — RAG Pipeline

Demonstrates retrieval-augmented generation against the sample program corpus,
including retrieval failure modes and their mitigations.

<!-- TODO main-session: expand teaching framing; tie back to NB 01 closing arc -->

## Setup

Adds the repo root to `sys.path`, loads environment variables, and imports the public RAG API.

<!-- TODO main-session: expand teaching framing -->

In [28]:
from __future__ import annotations
import os, sys, logging
from pathlib import Path

# Silence ChromaDB telemetry noise (posthog API mismatch; does not affect functionality)
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
# Silence ChromaDB "requested results > index size" warnings (harmless; Chroma auto-clamps)
logging.getLogger("chromadb").setLevel(logging.ERROR)

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env", override=False)

from src.rag import ingest, retrieve, RetrievedDocument
from src.llm import LLMClient

provider = os.getenv("LLM_PROVIDER", "anthropic")
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Provider: {provider} · Anthropic key present: {has_key}")


Provider: anthropic · Anthropic key present: True


## Ingest the corpus

Loads the five sample markdown documents, chunks them, embeds them, and persists the vector store.

<!-- TODO main-session: expand teaching framing -->

In [19]:
corpus_dir = repo_root / "data"
persist_dir = repo_root / "data" / "chroma_nb02"

result = ingest(
    corpus_dir=corpus_dir,
    persist_dir=persist_dir,
    chunk_size=500,
    chunk_overlap=50,
)

print(f"documents_loaded  : {result.documents_loaded}")
print(f"chunks_created    : {result.chunks_created}")
print(f"chunks_indexed    : {result.chunks_indexed}")
print(f"vector_store_path : {result.vector_store_path}")
print(f"embedding_model   : {result.embedding_model}")

documents_loaded  : 5
chunks_created    : 42
chunks_indexed    : 42
vector_store_path : c:\Users\narla\OneDrive\Desktop\TalentSprint\IISc_GenAI_C2\LLMOps\llmops-session\data\chroma_nb02
embedding_model   : sentence-transformers/all-MiniLM-L6-v2


## Baseline retrieval

Retrieve the top-5 chunks for a simple policy question and inspect scores + source priority.

<!-- TODO main-session: expand teaching framing -->


In [6]:
hits = retrieve(persist_dir, "What is the late submission policy?", k=5)

SEP = "-" * 60
for i, doc in enumerate(hits, 1):
    doc_id = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    print(f"Hit {i}")
    print(f"  document_id     : {doc_id}")
    print(f"  source_priority : {priority}")
    print(f"  score           : {doc.score:.4f}")
    print(f"  text (first 200): {doc.chunk.text[:200]!r}")
    print(SEP)


Hit 1
  document_id     : program_policy
  source_priority : 1
  score           : 0.9911
  text (first 200): '- Submissions up to **48 hours late** receive full credit with no\n  penalty if a brief note is added to the submission explaining the delay.\n- Submissions **48–168 hours late** (i.e. up to one week) r'
------------------------------------------------------------
Hit 2
  document_id     : assignment_guidelines
  source_priority : 2
  score           : 0.9769
  text (first 200): '## Late Submission Handling\n\nSee the **Program Policy** document, section *Late Submission Policy*, for\nthe authoritative rules. In summary: 48 hours grace with note → 10% penalty\nup to one week → not'
------------------------------------------------------------
Hit 3
  document_id     : faq
  source_priority : 5
  score           : 0.9441
  text (first 200): '**Q: What file format?**\nWhatever the assignment brief specifies. Default is `.ipynb` with outputs\nsaved for notebook assignments.\n\n**Q

## Notice the source_priority

The policy doc (`source_priority=1`) ranks at or near the top for a policy question — the retriever naturally surfaces the authoritative source. Later sections show when this breaks down.

<!-- TODO main-session: expand teaching framing -->


## Failure 1: parametric knowledge is not your policy

Cell 9 asks about a common program rule with no context — the model hedges because it correctly knows it doesn't have your specific policy. Cell 10 retrieves the exact rule. Vague training-data knowledge is not actionable; retrieved facts are.

<!-- TODO main-session: expand teaching framing -->

In [16]:
# Cell 9 — UNGROUNDED: ask about attendance norms with no retrieved context.
# Asking for the "most common" value forces a single number from parametric knowledge.
# Industry norm is typically 75%; our corpus sets 80% — close but wrong.

QUESTION = (
    "In a typical online AI engineering certification program, "
    "what is the standard minimum percentage of live sessions "
    "a participant must attend to remain in good standing?"
)

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no ANTHROPIC_API_KEY]")
else:
    client = LLMClient()
    result_ungrounded = client.complete(
        prompt=QUESTION,
        system=(
            "You are a helpful assistant. "
            "If the exact value varies by provider, give the single most commonly used figure."
        ),
        cache=False,
    )
    print("=== PARAMETRIC (no retrieval) ===")
    print(result_ungrounded.text)
    print(f"\n(This program's actual policy: 80% — see cell 10)")


=== PARAMETRIC (no retrieval) ===
I don't have reliable data on a universally "standard" minimum attendance requirement for online AI engineering certification programs, as this varies significantly by provider.

However, based on common practices across online professional certifications:

- **60-80% attendance** is a typical range for programs that track live session participation
- Some programs use **75% as a benchmark**
- Others are more flexible, requiring attendance only for specific mandatory sessions rather than all live sessions

**Important caveats:**
- Many online certification programs don't mandate live attendance at all—they offer recordings and asynchronous content
- Requirements differ between providers (Coursera, edX, university programs, corporate platforms, etc.)
- "Good standing" definitions vary (some focus on assignments/assessments rather than attendance)

If you're evaluating a specific program, I'd recommend checking their:
- Student handbook or program requir

In [17]:
# Cell 10 — GROUNDED: retrieve the attendance policy chunk, then ask the same question.
# The retrieved chunk explicitly states 80%; the grounded answer matches the doc exactly.

ATTENDANCE_QUESTION = (
    "What is the minimum percentage of live sessions a participant must attend "
    "to remain in good standing?"
)

grounded_hits = retrieve(persist_dir, "minimum attendance percentage live sessions", k=3)

context_parts = []
for rank, doc in enumerate(grounded_hits, 1):
    label = (
        f"[Source {rank}: {doc.chunk.metadata.document_id} "
        f"(priority={doc.chunk.metadata.source_priority}, score={doc.score:.4f})]"
    )
    context_parts.append(f"{label}\n{doc.chunk.text}")

context_block = "\n\n".join(context_parts)

grounded_prompt = (
    f"Use only the documents provided below to answer the question.\n\n"
    f"{context_block}\n\n"
    f"Question: {ATTENDANCE_QUESTION}"
)

print("=== RETRIEVED CONTEXT ===")
for part in context_parts:
    print(part[:300])
    print(SEP)

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no ANTHROPIC_API_KEY]")
else:
    client = LLMClient()
    result_grounded = client.complete(
        prompt=grounded_prompt,
        system=(
            "You are a helpful assistant. Answer only from the provided documents. "
            "If the answer is not in the documents, say so explicitly."
        ),
        cache=False,
    )
    print("\n=== GROUNDED ANSWER ===")
    print(result_grounded.text)


=== RETRIEVED CONTEXT ===
[Source 1: program_policy (priority=1, score=0.9800)]
## Attendance Expectations

Participants are expected to attend a minimum of **80% of live sessions**.
Attendance is tracked automatically by the learning platform.
------------------------------------------------------------
[Source 2: schedule (priority=4, score=0.9016)]
# Sample Program Schedule

> Synthetic document used for the LLM Ops teaching session. Dates and times
> are illustrative and do not refer to any real cohort.

## Weekly Session Pattern

| Day | Time (IST) | Session type |
|---|---|---|
| Tuesday | 19:
------------------------------------------------------------
[Source 3: faq (priority=5, score=0.8234)]
**Q: I joined late — can I still attend?**
Yes. Late joining is fine. Attendance is counted as long as you join within
the first 30 minutes of the session.

**Q: Can I attend on mobile?**
Theory sessions work fine on mobile. Hands-on sessions are not recomm
-------------------------------

## What just happened

Without retrieval the model hedged: *"60–80%, commonly 75%, check your handbook."* With retrieval it gave the exact rule: **80% of live sessions, tracked automatically.** A participant relying on the hedged answer might assume 75% and lose their standing. RAG replaces vague averages with authoritative facts.

<!-- TODO main-session: expand teaching framing -->

## Failure 2: wrong doc on top

Raw similarity retrieval picks whatever text is closest to the query — not whatever source is most authoritative.

<!-- TODO main-session: expand teaching framing -->


In [26]:
# Cell 13 — raw retrieval: wrong doc on top
# A query about assignment deadlines is lexically closest to the assignment_guidelines
# doc (source_priority=2), which edges out the authoritative program_policy (source_priority=1).
# The guidelines doc even defers to policy for the authoritative rules — but retrieval
# doesn't know that.

BAD_QUERY = "when is assignment 2 due"
SEP2 = "-" * 64

print(f"Query : {BAD_QUERY!r}  (k=3, raw similarity order)")
print(SEP2)
bad_hits = retrieve(persist_dir, BAD_QUERY, k=3)
for rank, doc in enumerate(bad_hits, 1):
    doc_id   = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    snippet  = doc.chunk.text[:160].replace("\n", " ")
    print(f"#{rank}  document_id={doc_id!r:28s}  source_priority={priority}  score={doc.score:.4f}")
    print(f"     {snippet!r}")
    print()

priorities_returned = [d.chunk.metadata.source_priority for d in bad_hits]
top_priority        = priorities_returned[0]
policy_in_results   = 1 in priorities_returned
if not policy_in_results:
    print(f"⚠  Policy doc (priority=1) is ABSENT from k=3 results — buried below lower-priority docs.")
elif top_priority > 1:
    print(f"⚠  Top result has source_priority={top_priority} — policy doc (priority=1) is NOT first.")
else:
    print(f"✓  Policy doc (priority=1) is already first — no failure to show; tune the query.")


Query : 'when is assignment 2 due'  (k=3, raw similarity order)
----------------------------------------------------------------
#1  document_id='assignment_guidelines'       source_priority=2  score=0.9556
     '# Sample Assignment Guidelines  > Synthetic document used for the LLM Ops teaching session.  ## Assignment Objective  Weekly assignments reinforce the concepts '

#2  document_id='support_process'             source_priority=3  score=0.8905
     '## Academic Doubt Process  1. Re-read the assignment brief and rubric 2. Check the FAQ document 3. If still unclear, post in `#academic-help` with a specific qu'

#3  document_id='faq'                         source_priority=5  score=0.8221
     '## Peer Discussion Rules  **Q: Can I discuss assignments with peers?** Discussing concepts is fine and encouraged. Sharing code or solutions is not permitted un'

⚠  Policy doc (priority=1) is ABSENT from k=3 results — buried below lower-priority docs.


In [29]:
# Cell 14 — fix: retrieve k=10, re-rank by source_priority, take top 3
# Fetching more candidates gives us the full authority ladder.
# Sorting by source_priority (lower = more authoritative) promotes the policy doc to #1.

print(f"Query : {BAD_QUERY!r}  (k=10 → sorted by source_priority asc)")
print(SEP2)
wide_hits  = retrieve(persist_dir, BAD_QUERY, k=10)
reranked   = sorted(wide_hits, key=lambda d: d.chunk.metadata.source_priority)
top3       = reranked[:3]

for rank, doc in enumerate(top3, 1):
    doc_id   = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    snippet  = doc.chunk.text[:160].replace("\n", " ")
    print(f"#{rank}  document_id={doc_id!r:28s}  source_priority={priority}  score={doc.score:.4f}")
    print(f"     {snippet!r}")
    print()

policy_rank = next(
    (i + 1 for i, d in enumerate(top3) if d.chunk.metadata.source_priority == 1), None
)
print(f"✓  Policy doc (source_priority=1) is now #{policy_rank} after re-ranking.")


Query : 'when is assignment 2 due'  (k=10 → sorted by source_priority asc)
----------------------------------------------------------------
#1  document_id='program_policy'              source_priority=1  score=0.9992
     '## Late Submission Policy  Assignments are due by **23:59 IST on the stated due date**.'

#2  document_id='assignment_guidelines'       source_priority=2  score=0.9903
     '# Sample Assignment Guidelines  > Synthetic document used for the LLM Ops teaching session.  ## Assignment Objective  Weekly assignments reinforce the concepts '

#3  document_id='support_process'             source_priority=3  score=0.9828
     '## Academic Doubt Process  1. Re-read the assignment brief and rubric 2. Check the FAQ document 3. If still unclear, post in `#academic-help` with a specific qu'

✓  Policy doc (source_priority=1) is now #1 after re-ranking.


## Source priority is the second-line defense

Re-ranking by `source_priority` costs one sort; it ensures policy-level documents are always promoted above summaries and FAQs even when embedding similarity disagrees.

<!-- TODO main-session: expand teaching framing -->
